<a href="https://colab.research.google.com/github/tasyriqomar/ColabMD-Edu_Protein-NA/blob/main/Docking_Protein_DNA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Install CondaColab to manage bioinformatics packages
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
# 2. Install the BioBB HADDOCK modules
!mamba install -y -c bioconda -c conda-forge biobb_haddock biobb_io biobb_pdb_tools nglview

In [3]:
# 3. Import modules for your pipeline
from biobb_io.api.pdb import pdb
from biobb_haddock.haddock.topology import topology
from biobb_haddock.haddock.rigid_body import rigid_body

In [ ]:
from google.colab import files

# 4. Upload your local Protein and DNA/RNA structures
print("Click 'Choose Files' to upload your protein and DNA/RNA .pdb files:")
uploaded = files.upload()

for filename in uploaded.keys():
    print(f'Successfully uploaded: "{filename}"')

In [ ]:
# Install the core HADDOCK3 engine
!pip install -q haddock3

In [ ]:
# 1. Install pdb-tools (if not already installed)
!pip install -q pdb-tools

# 2. Add chain 'B' to the DNA file and save it as DNA_fixed.pdb
!pdb_chain -B DNA.pdb > DNA_fixed.pdb

# 3. Create the DNA config file using the newly fixed PDB
dna_cfg = """
run_dir = "dna_topo"
mode = "local"
molecules = ["DNA_fixed.pdb"]

[topoaa]
"""
with open("dna_topo.cfg", "w") as f:
    f.write(dna_cfg)

# 4. Re-run the HADDOCK3 topology step for the DNA
!haddock3 dna_topo.cfg

print("Chain ID added. DNA Topology generation complete in 'dna_topo' directory.")

In [ ]:
# 1. Clear the old directory first
!rm -rf rigid_body_docking

# 2. Guarantee unique Chain IDs for both molecules
!pdb_chain -A RBD.pdb > RBD_fixed.pdb
!pdb_chain -B DNA.pdb > DNA_fixed.pdb

# 3. Create the configuration file using the fixed PDBs
docking_cfg = """
run_dir = "rigid_body_docking"
mode = "local"

# Feed the newly fixed files with unique chains
molecules = [
    "RBD_fixed.pdb",
    "DNA_fixed.pdb"
]

# Step 1: Topology generation
[topoaa]

# Step 2: Rigid-body docking
[rigidbody]
sampling = 20        # Number of models to generate
cmrest = true        # Enable center-of-mass restraints for ab-initio docking
"""

with open("docking.cfg", "w") as f:
    f.write(docking_cfg)

# 4. Run the HADDOCK3 pipeline
!haddock3 docking.cfg

print("Rigid-body docking complete! Results are in the 'rigid_body_docking' folder.")

In [ ]:
import gzip
import glob
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files
from IPython.display import display

# 1. Locate all compressed output models
output_folder = "rigid_body_docking/1_rigidbody/"
docked_poses = sorted(glob.glob(output_folder + "*.pdb.gz"))

if not docked_poses:
    raise FileNotFoundError(f"No .pdb.gz files found in {output_folder}")

pose_data = []

# 2. Extract HADDOCK score metrics from PDB file headers
for pose_path in docked_poses:
    pose_name = pose_path.split("/")[-1].replace(".pdb.gz", "")

    metrics = {
        'Pose': pose_name,
        'HADDOCK Score': None,
        'E_vdw (kcal/mol)': None,
        'E_elec (kcal/mol)': None,
        'E_desolv (kcal/mol)': None,
        'E_air (kcal/mol)': None,
        'BSA (Å²)': None
    }

    with gzip.open(pose_path, 'rt') as f:
        for line in f:
            if line.startswith("REMARK HADDOCK score:"):
                metrics['HADDOCK Score'] = float(line.split(":")[1].strip())
            elif line.startswith("REMARK vdw energy:"):
                metrics['E_vdw (kcal/mol)'] = float(line.split(":")[1].strip())
            elif line.startswith("REMARK elec energy:"):
                metrics['E_elec (kcal/mol)'] = float(line.split(":")[1].strip())
            elif line.startswith("REMARK desolv energy:"):
                metrics['E_desolv (kcal/mol)'] = float(line.split(":")[1].strip())
            elif line.startswith("REMARK air energy:"):
                metrics['E_air (kcal/mol)'] = float(line.split(":")[1].strip())
            elif line.startswith("REMARK buried surface area:"):
                metrics['BSA (Å²)'] = float(line.split(":")[1].strip())

            # Stop reading early once ATOM records begin
            if line.startswith("ATOM"):
                break

    pose_data.append(metrics)

# 3. Create and sort Pandas DataFrame by HADDOCK Score
df_scores = pd.DataFrame(pose_data)

# Sort by score if available, otherwise fallback to filename order
if df_scores['HADDOCK Score'].notnull().any():
    df_scores = df_scores.sort_values(by='HADDOCK Score', ascending=True).reset_index(drop=True)
else:
    print("Notice: HADDOCK score remarks not directly in headers; listed by pose index.")

# 4. Display the table in Colab
print(f"=== ALL {len(df_scores)} DOCKED POSES RANKED BY HADDOCK SCORE ===")
display(df_scores.style.hide(axis='index'))

# 5. Export to CSV and high-resolution PNG image
csv_filename = "haddock3_all_poses_scores.csv"
png_filename = "haddock3_all_poses_scores.png"

df_scores.to_csv(csv_filename, index=False)

# Render Matplotlib table image
fig, ax = plt.subplots(figsize=(10, len(df_scores) * 0.35 + 1.0), dpi=300)
ax.axis('off')
plt.title("HADDOCK3 Docking Scores & Energy Terms (All Poses)", fontsize=14, fontweight='bold', pad=15, color='#1e3d59')

table = ax.table(cellText=df_scores.values, colLabels=df_scores.columns, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.2, 1.4)

for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_facecolor('#1e3d59')
        cell.set_text_props(color='white', fontweight='bold')
    else:
        if row % 2 == 0:
            cell.set_facecolor('#f2f4f8')
        else:
            cell.set_facecolor('white')

plt.tight_layout()
plt.savefig(png_filename, bbox_inches='tight', dpi=300)
plt.close()

# 6. Trigger Downloads
files.download(csv_filename)
files.download(png_filename)
print("Downloaded score table CSV and PNG!")

In [ ]:
# 1. Install required packages
!pip install -q biopython pandas matplotlib

import gzip
import glob
import pandas as pd
import matplotlib.pyplot as plt
from Bio.PDB import PDBParser
from google.colab import files
from IPython.display import display, HTML

# 2. Locate the docked model
output_folder = "rigid_body_docking/1_rigidbody/"
docked_poses = sorted(glob.glob(output_folder + "*.pdb.gz"))

if not docked_poses:
    raise FileNotFoundError(f"No .pdb.gz files found in {output_folder}")

target_pose = docked_poses[0]
print(f"Analyzing contacts for: {target_pose}\n")

# 3. Parse structure
parser = PDBParser(QUIET=True)
with gzip.open(target_pose, 'rt') as handle:
    structure = parser.get_structure('docked_complex', handle)

model = structure[0]
chain_a = model['A'] # Protein
chain_b = model['B'] # DNA

heteroatoms = {'N', 'O', 'S', 'F', 'P'}
contact_cutoff = 4.0
hbond_cutoff = 3.5

hbond_list = []
contact_list = []

# 4. Extract atomic contacts and calculate distances
for atom_a in chain_a.get_atoms():
    if atom_a.element == 'H':
        continue
    for atom_b in chain_b.get_atoms():
        if atom_b.element == 'H':
            continue

        distance = atom_a - atom_b

        if distance <= contact_cutoff:
            res_a = atom_a.get_parent()
            res_b = atom_b.get_parent()

            res_a_name = f"{res_a.get_resname()}{res_a.get_id()[1]}"
            res_b_name = f"{res_b.get_resname()}{res_b.get_id()[1]}"

            contact_list.append({
                'Protein Residue (Chain A)': res_a_name,
                'DNA Residue (Chain B)': res_b_name,
                'Distance (Å)': round(float(distance), 2)
            })

            if distance <= hbond_cutoff and (atom_a.element in heteroatoms and atom_b.element in heteroatoms):
                hbond_list.append({
                    'Protein Residue': res_a_name,
                    'Protein Atom': atom_a.get_name(),
                    'DNA Residue': res_b_name,
                    'DNA Atom': atom_b.get_name(),
                    'Distance (Å)': round(float(distance), 2)
                })

# 5. Build Pandas DataFrames
df_hbonds = pd.DataFrame(hbond_list).drop_duplicates().sort_values(by='Distance (Å)')
df_contacts = pd.DataFrame(contact_list).drop_duplicates().sort_values(by='Distance (Å)')

# 6. Apply HTML styling for display inside Colab
def style_table(df, title):
    return df.style.set_caption(title).set_table_styles([
        {'selector': 'caption', 'props': [('color', '#1e3d59'), ('font-size', '16px'), ('font-weight', 'bold'), ('padding-bottom', '10px')]},
        {'selector': 'th', 'props': [('background-color', '#1e3d59'), ('color', 'white'), ('font-weight', 'bold'), ('text-align', 'center'), ('padding', '8px')]},
        {'selector': 'td', 'props': [('text-align', 'center'), ('padding', '8px')]},
        {'selector': 'tr:nth-child(even)', 'props': [('background-color', '#f2f4f8')]}
    ]).hide(axis='index')

print("=== POTENTIAL HYDROGEN BONDS / POLAR CONTACTS (< 3.5 Å) ===")
display(style_table(df_hbonds, "Hydrogen Bond & Polar Interaction Candidates"))

print("\n=== ALL INTERFACIAL RESIDUE PAIRS (< 4.0 Å) ===")
display(style_table(df_contacts, "Interfacial Residue Contact Pairs"))

# 7. Function to render DataFrame into high-resolution PNG using Matplotlib
def save_dataframe_as_png(df, title, filename):
    fig, ax = plt.subplots(figsize=(len(df.columns) * 2.5, len(df) * 0.4 + 1.2), dpi=300)
    ax.axis('off')

    # Title
    plt.title(title, fontsize=14, fontweight='bold', pad=15, color='#1e3d59')

    # Render Table
    table = ax.table(cellText=df.values, colLabels=df.columns, loc='center', cellLoc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.2, 1.5)

    # Custom cell styling
    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_facecolor('#1e3d59')
            cell.set_text_props(color='white', fontweight='bold')
        else:
            if row % 2 == 0:
                cell.set_facecolor('#f2f4f8')
            else:
                cell.set_facecolor('white')

    plt.tight_layout()
    plt.savefig(filename, bbox_inches='tight', dpi=300)
    plt.close()

# 8. Export high-res PNG images and CSV files
print("\nExporting high-resolution PNG images and CSV files...")

if not df_hbonds.empty:
    save_dataframe_as_png(df_hbonds, "Hydrogen Bond & Polar Interaction Candidates", 'hydrogen_bonds_table.png')
    df_hbonds.to_csv('hydrogen_bonds_table.csv', index=False)
    files.download('hydrogen_bonds_table.png')
    files.download('hydrogen_bonds_table.csv')

if not df_contacts.empty:
    save_dataframe_as_png(df_contacts, "Interfacial Residue Contact Pairs", 'interfacial_contacts_table.png')
    df_contacts.to_csv('interfacial_contacts_table.csv', index=False)
    files.download('interfacial_contacts_table.png')
    files.download('interfacial_contacts_table.csv')

print("Download complete!")

In [ ]:
# 1. Install py3Dmol (if you haven't already)
!pip install -q py3Dmol

import py3Dmol
import glob
import gzip
from IPython.display import display, HTML

# 1. Locate the compressed docked models (.pdb.gz) in the output folder
output_folder = "rigid_body_docking/1_rigidbody/"
docked_poses = sorted(glob.glob(output_folder + "*.pdb.gz"))

if docked_poses:
    # Select the first pose (rigidbody_1.pdb.gz) for visualization
    best_pose = docked_poses[0]
    print(f"Visualizing docked complex: {best_pose}")

    # 2. Decompress and read the PDB data directly in memory
    with gzip.open(best_pose, 'rt') as f:
        pdb_data = f.read()

    # 3. Initialize the py3Dmol viewer with high-res dimensions (1600x1200)
    view = py3Dmol.view(width=1600, height=1200)
    view.addModel(pdb_data, 'pdb')

    # 4. Apply styles to differentiate the molecules
    # Protein (Chain A) as a blue cartoon
    view.setStyle({'chain': 'A'}, {'cartoon': {'color': 'royalblue'}})

    # DNA (Chain B) as an orange cartoon with sticks for the nucleobases
    view.setStyle({'chain': 'B'}, {'cartoon': {'color': 'darkorange'}, 'stick': {'radius': 0.15}})

    # 5. Center the camera and display
    view.zoomTo()
    view.show()

    # 6. Append interactive JavaScript button to capture and download the WebGL canvas as a PNG
    download_js = """
    <button id="download_btn" style="
        background-color: #2e7d32;
        color: white;
        padding: 10px 20px;
        font-size: 14px;
        font-weight: bold;
        border: none;
        border-radius: 5px;
        cursor: pointer;
        margin-top: 10px;
    ">Download High-Res PNG</button>

    <script>
    document.getElementById('download_btn').onclick = function() {
        var canvases = document.getElementsByTagName('canvas');
        if (canvases.length > 0) {
            var canvas = canvases[canvases.length - 1];
            var imageURI = canvas.toDataURL("image/png").replace("image/png", "image/octet-stream");

            var link = document.createElement('a');
            link.download = 'docked_complex_highres.png';
            link.href = imageURI;
            document.body.appendChild(link);
            link.click();
            document.body.removeChild(link);
        } else {
            alert("Canvas not found. Please wait for the 3D model to finish rendering.");
        }
    };
    </script>
    """

    display(HTML(download_js))

else:
    print(f"No .pdb.gz files found in {output_folder}.")

In [ ]:
import gzip
import glob
import py3Dmol
from Bio.PDB import PDBParser
from IPython.display import display, HTML

# 1. Locate and parse the docked model
output_folder = "rigid_body_docking/1_rigidbody/"
docked_poses = sorted(glob.glob(output_folder + "*.pdb.gz"))

if not docked_poses:
    raise FileNotFoundError(f"No .pdb.gz files found in {output_folder}")

target_pose = docked_poses[0]

parser = PDBParser(QUIET=True)
with gzip.open(target_pose, 'rt') as handle:
    structure = parser.get_structure('docked_complex', handle)

# Decompress raw text data for py3Dmol
with gzip.open(target_pose, 'rt') as handle:
    raw_pdb_text = handle.read()

model = structure[0]
chain_a = model['A'] # Protein
chain_b = model['B'] # DNA

# 2. Extract residue IDs that form close contacts (< 4.0 Å)
contact_cutoff = 4.0
protein_interface_res = set()
dna_interface_res = set()

for atom_a in chain_a.get_atoms():
    if atom_a.element == 'H':
        continue
    for atom_b in chain_b.get_atoms():
        if atom_b.element == 'H':
            continue

        if (atom_a - atom_b) <= contact_cutoff:
            protein_interface_res.add(atom_a.get_parent().get_id()[1])
            dna_interface_res.add(atom_b.get_parent().get_id()[1])

# Convert residue sets to lists for py3Dmol selection
protein_res_list = list(protein_interface_res)
dna_res_list = list(dna_interface_res)

print(f"Protein interface residues (Chain A): {sorted(protein_res_list)}")
print(f"DNA interface residues (Chain B): {sorted(dna_res_list)}")

# 3. Render in py3Dmol with high-resolution canvas dimensions (1600x1200)
view = py3Dmol.view(width=1600, height=1200)
view.addModel(raw_pdb_text, 'pdb')

# Style full backbones as faded/semi-transparent cartoons
view.setStyle({'chain': 'A'}, {'cartoon': {'color': 'royalblue', 'opacity': 0.4}})
view.setStyle({'chain': 'B'}, {'cartoon': {'color': 'darkorange', 'opacity': 0.4}})

# Highlight Protein interface residues (Cyan sticks)
view.addStyle(
    {'chain': 'A', 'resi': protein_res_list},
    {'stick': {'color': 'cyan', 'radius': 0.2}, 'cartoon': {'color': 'royalblue', 'opacity': 1.0}}
)

# Highlight DNA interface residues (Yellow/Orange sticks)
view.addStyle(
    {'chain': 'B', 'resi': dna_res_list},
    {'stick': {'color': 'yellow', 'radius': 0.2}, 'cartoon': {'color': 'darkorange', 'opacity': 1.0}}
)

# Zoom to the protein interface for an up-close view
view.zoomTo({'chain': 'A', 'resi': protein_res_list})
view.show()

# 4. Add interactive download button using JavaScript
download_js = """
<button id="download_btn" style="
    background-color: #2e7d32;
    color: white;
    padding: 10px 20px;
    font-size: 14px;
    font-weight: bold;
    border: none;
    border-radius: 5px;
    cursor: pointer;
    margin-top: 10px;
">Download High-Res PNG</button>

<script>
document.getElementById('download_btn').onclick = function() {
    var canvases = document.getElementsByTagName('canvas');
    if (canvases.length > 0) {
        var canvas = canvases[canvases.length - 1];
        var imageURI = canvas.toDataURL("image/png").replace("image/png", "image/octet-stream");

        var link = document.createElement('a');
        link.download = 'docked_interface_highres.png';
        link.href = imageURI;
        document.body.appendChild(link);
        link.click();
        document.body.removeChild(link);
    } else {
        alert("Canvas not found. Please wait for the 3D model to finish rendering.");
    }
};
</script>
"""

display(HTML(download_js))

In [ ]:
import gzip
import glob
import py3Dmol
from Bio.PDB import PDBParser
from IPython.display import display, HTML

# 1. Locate and parse the docked model
output_folder = "rigid_body_docking/1_rigidbody/"
docked_poses = sorted(glob.glob(output_folder + "*.pdb.gz"))

if not docked_poses:
    raise FileNotFoundError(f"No .pdb.gz files found in {output_folder}")

target_pose = docked_poses[0]

parser = PDBParser(QUIET=True)
with gzip.open(target_pose, 'rt') as handle:
    structure = parser.get_structure('docked_complex', handle)

with gzip.open(target_pose, 'rt') as handle:
    raw_pdb_text = handle.read()

model = structure[0]
chain_a = model['A'] # Protein
chain_b = model['B'] # DNA

# 2. Identify contact atom pairs and residue lists
contact_cutoff = 4.0
protein_interface_res = set()
dna_interface_res = set()
labeled_atoms = []

for atom_a in chain_a.get_atoms():
    if atom_a.element == 'H':
        continue
    for atom_b in chain_b.get_atoms():
        if atom_b.element == 'H':
            continue

        if (atom_a - atom_b) <= contact_cutoff:
            res_a = atom_a.get_parent()
            res_b = atom_b.get_parent()

            protein_interface_res.add(res_a.get_id()[1])
            dna_interface_res.add(res_b.get_id()[1])

            labeled_atoms.append((atom_a, res_a, 'A'))
            labeled_atoms.append((atom_b, res_b, 'B'))

protein_res_list = list(protein_interface_res)
dna_res_list = list(dna_interface_res)

# 3. Render in py3Dmol with high-resolution canvas settings
# Higher width/height gives a crisp, high-res export
view = py3Dmol.view(width=1600, height=1200)
view.addModel(raw_pdb_text, 'pdb')

# Style background cartoons
view.setStyle({'chain': 'A'}, {'cartoon': {'color': 'royalblue', 'opacity': 0.3}})
view.setStyle({'chain': 'B'}, {'cartoon': {'color': 'darkorange', 'opacity': 0.3}})

# Highlight interface sticks
view.addStyle(
    {'chain': 'A', 'resi': protein_res_list},
    {'stick': {'color': 'cyan', 'radius': 0.2}, 'cartoon': {'color': 'royalblue', 'opacity': 1.0}}
)
view.addStyle(
    {'chain': 'B', 'resi': dna_res_list},
    {'stick': {'color': 'yellow', 'radius': 0.2}, 'cartoon': {'color': 'darkorange', 'opacity': 1.0}}
)

# Add text labels to interacting atoms
seen_atoms = set()
for atom, res, chain_id in labeled_atoms:
    atom_key = (chain_id, res.get_id()[1], atom.get_name())
    if atom_key in seen_atoms:
        continue
    seen_atoms.add(atom_key)

    label_text = f"{res.get_resname()}{res.get_id()[1]}:{atom.get_name()}"
    pos = atom.get_coord()
    bg_color = '#1e3d59' if chain_id == 'A' else '#d96459'

    view.addLabel(
        label_text,
        {
            'position': {'x': float(pos[0]), 'y': float(pos[1]), 'z': float(pos[2])},
            'backgroundColor': bg_color,
            'fontColor': 'white',
            'fontSize': 12,
            'borderThickness': 0.5,
            'backgroundOpacity': 0.8
        }
    )

view.zoomTo({'chain': 'A', 'resi': protein_res_list})

# Render the 3D canvas
view.show()

# 4. Inject JavaScript button to download the canvas as a PNG
download_js = """
<button id="download_btn" style="
    background-color: #4CAF50;
    color: white;
    padding: 10px 20px;
    font-size: 14px;
    font-weight: bold;
    border: none;
    border-radius: 5px;
    cursor: pointer;
    margin-top: 10px;
">Download High-Res PNG</button>

<script>
document.getElementById('download_btn').onclick = function() {
    // Locate the WebGL canvas created by py3Dmol
    var canvases = document.getElementsByTagName('canvas');
    if (canvases.length > 0) {
        var canvas = canvases[canvases.length - 1]; // Target active viewer canvas

        // Convert canvas buffer to PNG data URL
        var imageURI = canvas.toDataURL("image/png").replace("image/png", "image/octet-stream");

        // Trigger browser download
        var link = document.createElement('a');
        link.download = 'docked_interface_labeled.png';
        link.href = imageURI;
        document.body.appendChild(link);
        link.click();
        document.body.removeChild(link);
    } else {
        alert("Canvas not found. Please wait for the 3D model to render completely.");
    }
};
</script>
"""

display(HTML(download_js))

In [ ]:
import gzip
import glob
from google.colab import files

# 1. Locate the top docked model (.pdb.gz)
output_folder = "rigid_body_docking/1_rigidbody/"
docked_poses = sorted(glob.glob(output_folder + "*.pdb.gz"))

if docked_poses:
    target_pose = docked_poses[0]
    output_pdb_name = "docked_complex_best.pdb"

    # 2. Decompress and save as a standard .pdb file
    with gzip.open(target_pose, 'rt') as f_in:
        with open(output_pdb_name, 'w') as f_out:
            f_out.write(f_in.read())

    print(f"Decompressed {target_pose} to {output_pdb_name}")

    # 3. Trigger automatic download to your local machine
    files.download(output_pdb_name)
else:
    print(f"No .pdb.gz files found in {output_folder}")